In [1]:
import os
from dotenv import load_dotenv

os.chdir("D:/my-agent-project")
print("当前工作目录:", os.getcwd())

# 如果你把文件重命名为 .env，就用这一行：
load_dotenv()

# 如果你保留文件名 dotenv.env，就用这一行：
# load_dotenv("dotenv.env")

print("API Key 是否加载成功:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("API Key 前10位:", os.getenv("DEEPSEEK_API_KEY")[:10] if os.getenv("DEEPSEEK_API_KEY") else "未加载")

当前工作目录: D:\my-agent-project
API Key 是否加载成功: True
API Key 前10位: sk-159bed4


In [2]:
from langchain_openai import ChatOpenAI

# 使用已加载的环境变量
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),   # 注意参数名是 api_key
    base_url="https://api.deepseek.com/v1",  # 注意参数名是 base_url
    temperature=0.7
)

response = llm.invoke("请用一句话解释什么是 AI Agent")
print(response.content)

AI Agent是能够自主感知环境、制定决策并执行行动以实现特定目标的智能实体。


In [3]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver

# 1. 加载 API 密钥（确保 .env 文件在项目根目录）
load_dotenv()

# 2. 初始化模型（使用 DeepSeek）
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0.7
)

# 3. 定义 Agent 节点：接收当前状态，调用模型，返回新消息
def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    # 返回 {"messages": [response]} 会自动追加到消息列表
    return {"messages": [response]}

# 4. 构建状态图
workflow = StateGraph(MessagesState)   # 使用内置的 MessagesState
workflow.add_node("agent", call_model)
workflow.add_edge(START, "agent")
workflow.add_edge("agent", END)

# 5. 编译图并注入记忆（使用内存检查点）
memory = MemorySaver()                 # 记忆保存器
graph = workflow.compile(checkpointer=memory)

# 6. 配置会话 ID（相同的 ID 会共享记忆）
config = {"configurable": {"thread_id": "user-123"}}

# 第一轮对话：告诉名字
print("=== 第一轮 ===")
response1 = graph.invoke(
    {"messages": [("user", "你好，我叫张三")]},
    config=config
)
print("AI:", response1["messages"][-1].content)

# 第二轮对话：询问名字（模型应该记住）
print("\n=== 第二轮 ===")
response2 = graph.invoke(
    {"messages": [("user", "我刚才说我叫什么名字？")]},
    config=config
)
print("AI:", response2["messages"][-1].content)

=== 第一轮 ===
AI: 你好，张三！很高兴认识你！😊

我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要帮助，或者只是想聊聊天，我都很乐意陪你！

有什么我可以帮你的吗？比如：
- 解答疑惑
- 提供建议
- 聊聊兴趣爱好
- 或者随便聊聊日常

请随时告诉我你的需求！🌟

=== 第二轮 ===
AI: 你刚才说你的名字是**张三**！😊 我记住了～
